In [ ]:
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

In [ ]:
import sys
import os
sys.path.append('/content/gdrive/MyDrive/???/')
os.chdir('/content/gdrive/MyDrive/???/')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Load your time series data
df = pd.read_csv('bike_day.csv')
df.head()

In [ ]:
df.dtypes

#### String Time Format Code List: https://strftime.org/

In [ ]:
df['dteday'] = pd.to_datetime(df['dteday'], format='%Y-%m-%d')  # Match format according to the data pattern
df = df.set_index('dteday')
df.head()

In [ ]:
df.columns

### SARIMA Model

In [ ]:
train = df.loc[df.index <= '2012-11-01']
test = df.loc[df.index > '2012-11-01']
forecast_steps = len(test)

TARGET = 'cnt'
y_train = train[TARGET]
y_test = test[TARGET]

In [ ]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Set frequeuncy of time index
train.index = pd.DatetimeIndex(train.index, freq='D')
train.index.inferred_freq

log_train = np.log(train[TARGET])
# Define the model
model = SARIMAX(log_train,
                order=(6, 1, 0),
                seasonal_order=(2, 0, 0, 7),
                )

# Fit the model
model_fit = model.fit()

# Make predictions
log_forecast = model_fit.forecast(steps=forecast_steps)
forecast = np.exp(log_forecast)

# Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(test[TARGET], forecast))
mae = mean_absolute_error(test[TARGET], forecast)
print("RMSE:", rmse, ", MAE :", mae)

In [ ]:
result = df.copy()
test['arima_prediction'] = forecast
result = result.merge(test['arima_prediction'], how='left', left_index=True, right_index=True)

plt.figure(figsize=(15,5))
plt.plot(result[TARGET])
plt.plot(test['arima_prediction'], '-.')
plt.legend(['Actual', 'ARIMA Forecast'])
plt.title('Actual vs. ARIMA Forecast')
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(test[TARGET])
plt.plot(test['arima_prediction'], '-.')
plt.legend(['Actual', 'ARIMA Forecast'])
plt.title('Actual vs. ARIMA Forecast')
plt.show()

### Regression with Lags

In [ ]:
value_var = 'cnt'

In [ ]:
def create_lags(data, lags=12):
    df_lag = pd.DataFrame(index=data.index)

    for i in range(1, lags+1):
        df_lag[f'lag_{i}'] = data.shift(i)

    df_lag['y'] = data
    return df_lag.dropna()

data = create_lags(df[value_var], lags=12)
data.head()

In [ ]:
split = int(len(data)*0.7)

train = data.iloc[:split]
test  = data.iloc[split:]

X_train, y_train = train.drop(columns='y'), train['y']
X_test,  y_test  = test.drop(columns='y'),  test['y']

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled, y_train)

In [ ]:
reg_predictions = model.predict(X_test_scaled)


from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test, reg_predictions)
rmse = np.sqrt(mean_squared_error(y_test, reg_predictions))

print("RMSE:", rmse, "MAE:", mae)

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(y_test, label="Actual")
plt.plot(y_test.index, reg_predictions, '-.', label="Regression with Lags")
plt.legend(['Actual', 'Regression with Lags'])
plt.title('Actual vs. Regression with Lags')
plt.show()

### RNN with Lags

In [ ]:
value_var = 'cnt'

In [ ]:
from sklearn.preprocessing import StandardScaler

data = df[value_var].values.astype('float32')
data = data.reshape(-1,1)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data).flatten()

In [ ]:
def create_sequences(data, window=12):
    X, y = [], []
    for i in range(len(data)-window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, window=14)

In [ ]:
X = X.reshape((X.shape[0], X.shape[1], 1))
split = int(len(X)*0.7)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn = Sequential([
    SimpleRNN(32, activation='tanh', input_shape=(window,1)),
    Dense(1)
])

rnn.compile(optimizer='adam', loss='mse')
rnn.fit(X_train, y_train, epochs=100, verbose=1)

In [ ]:
rnn_predictions = rnn.predict(X_test)

y_test_inv  = scaler.inverse_transform(y_test.reshape(-1,1))
rnn_predictions_inv  = scaler.inverse_transform(rnn_predictions)

# Evaluation
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test_inv, rnn_predictions_inv))
mae = mean_absolute_error(y_test_inv, rnn_predictions_inv)
print("RMSE:", rmse, ", MAE :", mae)

In [ ]:
test.head()

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(y_test_inv, label="Actual")
plt.plot(rnn_predictions_inv, '-.', label="RNN with Lags")
plt.legend(['Actual', 'RNN with Lags'])
plt.title('Actual vs. RNN with Lags')
plt.show()

### LSTM with Lags

In [ ]:
from tensorflow.keras.layers import LSTM

lstm = Sequential([
    LSTM(32, activation='tanh', input_shape=(window,1)),
    Dense(1)
])

lstm.compile(optimizer='adam', loss='mse')

lstm.fit(X_train, y_train, epochs=100, verbose=1)

In [ ]:
lstm_predictions = lstm.predict(X_test)

y_test_inv  = scaler.inverse_transform(y_test.reshape(-1,1))
lstm_predictions_inv  = scaler.inverse_transform(lstm_predictions)

# Evaluation
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test_inv, lstm_predictions_inv))
mae = mean_absolute_error(y_test_inv, lstm_predictions_inv)
print("RMSE:", rmse, ", MAE :", mae)

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(y_test_inv, label="Actual")
plt.plot(rnn_predictions_inv, '-.', label="LSTM with Lags")
plt.legend(['Actual', 'LSTM with Lags'])
plt.title('Actual vs. LSTM with Lags')
plt.show()

### RNN with features

In [ ]:
train = df.loc[df.index <= '2012-11-01']
test = df.loc[df.index > '2012-11-01']
forecast_steps = len(test)

FEATURES = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday',
            'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
TARGET = 'cnt'

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense


# Prepare your time series data (train_x, train_y)
train_x = np.array(X_train)  # input sequences
train_y = np.array(y_train)  # target values

# Reshape the data for RNN
train_x = train_x.reshape((train_x.shape[0], train_x.shape[1], 1))

# Build the RNN model
model = Sequential()
model.add(SimpleRNN(50, activation='relu', input_shape=(train_x.shape[1], 1)))
model.add(Dense(1))

# Compile and fit the model
model.compile(optimizer='adam', loss='mse')
model.fit(train_x, train_y, epochs=20, verbose=1)

In [ ]:
# Predict values
test_x = np.array(X_test)  # input sequences
test_y = np.array(y_test)  # target values

predictions = model.predict(test_x)

# Evaluation
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(test_y, predictions))
mae = mean_absolute_error(test_y, predictions)
print("RMSE:", rmse, ", MAE :", mae)

In [ ]:
result = df.copy()
test['rnn_prediction'] = predictions
result = result.merge(test['rnn_prediction'], how='left', left_index=True, right_index=True)

plt.figure(figsize=(15,5))
plt.plot(result[[TARGET]])
plt.plot(test['rnn_prediction'], '-.')
plt.legend(['Actual', 'RNN with Features'])
plt.title('Actual vs. RNN with Features')
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(test[TARGET])
plt.plot(test['rnn_prediction'], '-.', label='RNN with Features')
plt.legend(['Actual', 'RNN with Features'])
plt.title('Actual vs. RNN with Features')
plt.show()

### LSTM with features

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Reshape for LSTM
train_x = train_x.reshape((train_x.shape[0], train_x.shape[1], 1))

# Build the LSTM model
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(train_x.shape[1], 1)))
model.add(Dense(1))

# Compile and fit the model
model.compile(optimizer='adam', loss='mse')
model.fit(train_x, train_y, epochs=20, verbose=1)

In [ ]:
# Predict values
test_x = np.array(X_test)  # input sequences
test_y = np.array(y_test)  # target values

predictions = model.predict(test_x)

# Evaluation
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(test_y, predictions))
mae = mean_absolute_error(test_y, predictions)
print("RMSE:", rmse, ", MAE :", mae)

In [ ]:
result = df.copy()
test['lstm_prediction'] = predictions
result = result.merge(test['lstm_prediction'], how='left', left_index=True, right_index=True)

plt.figure(figsize=(15,5))
plt.plot(result[[TARGET]])
plt.plot(test['lstm_prediction'], '-.')
plt.legend(['Actual', 'LSTM with Features'])
plt.title('Actual vs. LSTM with Features')
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(test[TARGET])
plt.plot(test['lstm_prediction'], '-.')
plt.legend(['Actual', 'LSTM with Features'])
plt.title('Actual vs. LSTM with Features')
plt.show()